# **MÓDULO 35 - Cross Validation**

Nesta tarefa, você trabalhará com uma base de dados que contém informações sobre variáveis ambientais coletadas para a detecção de incêndios. O objetivo é utilizar técnicas de validação cruzada (cross-validation) para avaliar a performance de um modelo de classificação na previsão da ocorrência de um incêndio com base nas variáveis fornecidas.


Descrição da Base de Dados
A base de dados contém as seguintes variáveis:

Unnamed:0: Índice (não é uma variável útil para o modelo)

UTC: Tempo em Segundos UTC

Temperature[C]: Temperatura do Ar (em graus Celsius)

Humidity[%]: Umidade do Ar (em porcentagem)

TVOC[ppb]: Total de Compostos Orgânicos Voláteis (medido em partes por bilhão)

eCO2[ppm]: Concentração equivalente de CO2 (medido em partes por milhão)

Raw H2: Hidrogênio molecular bruto, não compensado

Raw Ethanol: Etanol gasoso bruto

Pressure[hPA]: Pressão do Ar (em hectopascais)

PM1.0: Material particulado de tamanho < 1,0 µm

PM2.5: Material particulado de tamanho >1,0 µm e < 2,5 µm

NC0.5: Concentração numérica de material particulado de tamanho < 0,5 µm

NC1.0: Concentração numérica de material particulado de tamanho 0,5 µm < 1,0 µm

NC2.5: Concentração numérica de material particulado de tamanho 1,0 µm < 2,5 µm

CNT: Contador de amostras


E a variável alvo:

Fire Alarm: Indicador binário de incêndio (1 se houver incêndio, 0 caso contrário)

O objetivo desta tarefa é aplicar a técnica de validação cruzada (cross-validation) para avaliar a performance de um modelo de classificação. A validação cruzada ajudará a garantir que o modelo seja avaliado de maneira robusta e generalize bem para dados não vistos.

In [ ]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

# 1 - Carregue a base de dados, verifique os tipos de dados e também se há presença de dados faltantes ou nulos.

In [2]:
df = pd.read_csv("Cientista de dados M35 - smoke_detection_iot.csv", delimiter=",")

In [3]:
df.head()

,Unnamed: 0,UTC,Temperature[C],Humidity[%],TVOC[ppb],eCO2[ppm],Raw H2,Raw Ethanol,Pressure[hPa],PM1.0,PM2.5,NC0.5,NC1.0,NC2.5,CNT,Fire Alarm
0,0,1654733331,20.000,57.36,0,400,12306,18520,939.735,0.0,0.0,0.0,0.0,0.0,0,0
1,1,1654733332,20.015,56.67,0,400,12345,18651,939.744,0.0,0.0,0.0,0.0,0.0,1,0
2,2,1654733333,20.029,55.96,0,400,12374,18764,939.738,0.0,0.0,0.0,0.0,0.0,2,0
3,3,1654733334,20.044,55.28,0,400,12390,18849,939.736,0.0,0.0,0.0,0.0,0.0,3,0
4,4,1654733335,20.059,54.69,0,400,12403,18921,939.744,0.0,0.0,0.0,0.0,0.0,4,0


In [4]:
df = df.drop(columns=["Unnamed: 0"])

In [5]:
df.rename(columns={'Fire Alarm': 'Fire_Alarm'}, inplace=True)

In [6]:
df.describe()

,UTC,Temperature[C],Humidity[%],TVOC[ppb],eCO2[ppm],Raw H2,Raw Ethanol,Pressure[hPa],PM1.0,PM2.5,NC0.5,NC1.0,NC2.5,CNT,Fire_Alarm
count,6.263000e+04,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000
mean,1.654792e+09,15.970424,48.539499,1942.057528,670.021044,12942.453936,19754.257912,938.627649,100.594309,184.467770,491.463608,203.586487,80.049042,10511.386157,0.714626
std,1.100025e+05,14.359576,8.865367,7811.589055,1905.885439,272.464305,609.513156,1.331344,922.524245,1976.305615,4265.661251,2214.738556,1083.383189,7597.870997,0.451596
min,1.654712e+09,-22.010000,10.740000,0.000000,400.000000,10668.000000,15317.000000,930.852000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.654743e+09,10.994250,47.530000,130.000000,400.000000,12830.000000,19435.000000,938.700000,1.280000,1.340000,8.820000,1.384000,0.033000,3625.250000,0.000000
50%,1.654762e+09,20.130000,50.150000,981.000000,400.000000,12924.000000,19501.000000,938.816000,1.810000,1.880000,12.450000,1.943000,0.044000,9336.000000,1.000000
75%,1.654778e+09,25.409500,53.240000,1189.000000,438.000000,13109.000000,20078.000000,939.418000,2.090000,2.180000,14.420000,2.249000,0.051000,17164.750000,1.000000
max,1.655130e+09,59.930000,75.200000,60000.000000,60000.000000,13803.000000,21410.000000,939.861000,14333.690000,45432.260000,61482.030000,51914.680000,30026.438000,24993.000000,1.000000


In [7]:
df.isnull().sum()

UTC               0
Temperature[C]    0
Humidity[%]       0
TVOC[ppb]         0
eCO2[ppm]         0
Raw H2            0
Raw Ethanol       0
Pressure[hPa]     0
PM1.0             0
PM2.5             0
NC0.5             0
NC1.0             0
NC2.5             0
CNT               0
Fire_Alarm        0
dtype: int64

In [8]:
df.dtypes

UTC                 int64
Temperature[C]    float64
Humidity[%]       float64
TVOC[ppb]           int64
eCO2[ppm]           int64
Raw H2              int64
Raw Ethanol         int64
Pressure[hPa]     float64
PM1.0             float64
PM2.5             float64
NC0.5             float64
NC1.0             float64
NC2.5             float64
CNT                 int64
Fire_Alarm          int64
dtype: object

Para a coluna Fire Alarm, por conta do espaçamento talvez seja util renomear o nome da coluna utilizando:

df.rename(columns={'Fire Alarm': 'Fire_Alarm'}, inplace=True)

# 2 - Para essa base, onde você realizará as previsões de fire alarm, qual modelo de machine learning você aplicará? Justifique.

Para essa base de dados, o modelo escolhido foi a Regressão Logística, pois o objetivo é realizar uma classificação binária, em que o modelo deve prever se ocorrerá ou não um incêndio, representado pela variável 'Fire Alarm'. Esse tipo de problema é adequado para a Regressão Logística, pois o algoritmo estima a probabilidade de uma determinada observação pertencer a uma das duas classes, permitindo posteriormente classificá-la como ocorrência ou não ocorrência de incêndio.

A escolha desse modelo também se deve à sua simplicidade e facilidade de interpretação, sendo uma boa opção para estabelecer um modelo de classificação inicial. O parâmetro 'max_iter=1000' foi utilizado para aumentar o número máximo de iterações disponíveis para o algoritmo durante o treinamento. Isso ajuda a garantir que o modelo tenha tempo suficiente para convergir para uma solução adequada, evitando problemas em que o algoritmo atinge o limite de iterações antes de encontrar os melhores parâmetros.

# 3 - Separe a base em Y e X e já rode a instância do modelo que você utilizará.

In [9]:
X = df.drop('Fire_Alarm', axis=1)
y = df['Fire_Alarm']

In [10]:
modelo = LogisticRegression(max_iter=1000)

# 4 - Defina o número de Folds e rode o modelo com a validação cruzada.

In [11]:
folds = 5

In [16]:
crossvalidation = KFold(n_splits=folds, shuffle=True, random_state=42)

In [17]:
modelo_final = cross_val_score(modelo, X, y, cv = folds)

# 5 - Avalie a pontuação de cada modelo e ao final a validação final da média.

In [21]:
pontuacoes = cross_val_score(modelo, X, y, cv=crossvalidation)

print(f"Pontuações por fold: {pontuacoes}")

Pontuações por fold: [0.97668849 0.98626856 0.97844483 0.97700782 0.9776465 ]


In [22]:
print((modelo_final.mean()))

0.9619511416254192


Foi aplicada a técnica de Cross-Validation, dividindo os dados em 5 folds. As pontuações obtidas em cada fold foram:
- Fold 1: 0.9766
- Fold 2: 0.9862
- Fold 3: 0.9784
- Fold 4: 0.9770
- Fold 5: 0.9776

A média das pontuações foi de 0.9619, indicando que o modelo apresentou um desempenho bastante consistente entre os diferentes subconjuntos dos dados.

A utilização da validação cruzada com 5 folds permite avaliar o modelo de maneira mais robusta. Em vez de avaliar o modelo utilizando apenas uma divisão entre treinamento e teste, os dados são divididos em cinco partes e o modelo é treinado e avaliado várias vezes. Os resultados obtidos foram próximos entre os folds, apresentando uma média de aproximadamente 96,19%. Isso indica que o modelo apresentou um desempenho elevado e relativamente consistente nas diferentes divisões da base.